In [7]:
import warnings
import torch
import torch.nn.functional as AF
from torch import Tensor
from torch.testing import assert_close

print("PyTorch version:", torch.__version__)

PyTorch version: 2.13.0+cpu


In [9]:
import torch

x = torch.arange(1.0, 5, requires_grad=True)
y = torch.arange(5.0, 9, requires_grad=True)
q = x.dot(y)
z = q.sin()
print('z.requires_grad:', z.requires_grad)
print('z.grad_fn:', z.grad_fn.name())
print('q.grad_fn:', q.grad_fn.name())
print('x.grad_fn:', x.grad_fn)
print('y.grad_fn:', y.grad_fn)
print('x.grad:', x.grad)
print('y.grad:', y.grad)


z.requires_grad: True
z.grad_fn: SinBackward0
q.grad_fn: DotBackward0
x.grad_fn: None
y.grad_fn: None
x.grad: None
y.grad: None


In [10]:
# backward(): 从输出往回传梯度
z.backward()
print('x.grad:', x.grad)
print('y.grad:', y.grad)
# 验算梯度是否一致
expected_grad_x = y * x.dot(y).cos()
expected_grad_y = x * x.dot(y).cos()
assert_close(x.grad, expected_grad_x)
assert_close(y.grad, expected_grad_y)

x.grad: tensor([3.1666, 3.7999, 4.4332, 5.0666])
y.grad: tensor([0.6333, 1.2666, 1.9000, 2.5333])


In [13]:
"""
    标量输出默认从1开始反传,非标量输出需要我们自己给出输出端传回的数据
"""

# 非标量为什么不能直接backword()?
x = torch.arange(1.0, 5, requires_grad=True)
y = torch.arange(5.0, 9, requires_grad=True)
z = x.outer(y)

try:
    z.backward()
except RuntimeError as err:
    print('RuntimeError', err)

Z = x.outer(y)
z.backward(gradient=torch.ones_like(Z))
# 非标量不能直接backword() 因为 要明确输出端回传的梯度是什么
print('x.grad:', x.grad)
print('y.grad:', y.grad)

# 等价将所有的元素求和,然后对标量调用backward()
x = torch.arange(1.0, 5, requires_grad=True)
y = torch.arange(5.0, 9, requires_grad=True)

Z1 = x.outer(y)
loss = Z1.sum()
loss.backward()

print('x.grad:', x.grad)
print('y.grad:', y.grad)


RuntimeError grad can be implicitly created only for scalar outputs
x.grad: tensor([26., 26., 26., 26.])
y.grad: tensor([10., 10., 10., 10.])
x.grad: tensor([26., 26., 26., 26.])
y.grad: tensor([10., 10., 10., 10.])


In [20]:
"""
    高阶导数: 让求导的过程页变成计算的一部分
    -  需要在同一次前向计算得到的图上反复坐反向传播,可以设置retain_graph=True

"""
x = torch.tensor(2.0, requires_grad=True)
y = torch.tensor(4.0, requires_grad=True)
z = x.mul(y).sin()

dzdx, dzdy = torch.autograd.grad(z, (x, y), create_graph=True)
print('dz/dx:', dzdx)
print('dz/dy:', dzdy)
(d2zdx2,) = torch.autograd.grad(dzdx, x, retain_graph=True)
(d2zdy2,) = torch.autograd.grad(dzdy, y)

print('d2z/dx2:', d2zdx2)
print('d2z/dy:', d2zdy2)

dz/dx: tensor(-0.5820, grad_fn=<MulBackward0>)
dz/dy: tensor(-0.2910, grad_fn=<MulBackward0>)
d2z/dx2: tensor(-15.8297, grad_fn=<MulBackward0>)
d2z/dy: tensor(-3.9574)


In [25]:
"""
    VJP 和 JVP : 真正计算的不是完整的Jacobian
    - 正向模式(vjp): 适合输出维度小,输入维度大的情况,比如神经网络训练
    - 反向模式(jvp): 适合输入维度大,输出维度小的情况,比如科学计算

"""
from torch.func import vjp, jvp


# 向量-雅可比积

def vjp_func(x: Tensor, y: Tensor):
    return x.dot(y).sin()


x = torch.arange(1.0, 5)
y = torch.arange(5.0, 9)
output = vjp(vjp_func, x, y)

print('func(x,y)', output[0])
print('VJP output', output[1])


# JVP: 雅可比-向量积

def jvp_func(x: Tensor, y: Tensor):
    return x.dot(y).sin()


x = torch.arange(1.0, 5)
y = torch.arange(5.0, 9)
u_x = torch.full_like(x, 0.1)
u_y = torch.full_like(y, 0.2)

output = jvp(jvp_func, (x, y), (u_x, u_y))

print("func(x,y)", output[0])
print("JVP output", output[1])




func(x,y) tensor(0.7739)
VJP output <function _vjp_with_argnums.<locals>.wrapper at 0x000001B5F421FA00>
func(x,y) tensor(0.7739)
JVP output tensor(2.9133)


In [34]:
"""
    反向传播的几个常见问题
     - 重复调用backward()
     - 梯度会累计
     - 中间节点默认不会保存梯度信息
     - 原地操作可能破坏反向传播
"""

x = torch.arange(1.0, 5, requires_grad=True)
y = torch.arange(5.0, 9, requires_grad=True)
# 重复调用backward()
z = x.dot(y).sin()
# z.backward()


# try:
#     z.backward()
# except RuntimeError as err:
#     print('RuntimeError', err)
# 同一个图上多次传播,可以使用retain_graph = True

z.backward(retain_graph=True)
z.backward()



#  梯度会累计
#  训练循环的时候,使用zero_grad(),把之前的梯度清零
x.grad = None
print('x.grad:', x.grad)

z1 = x.dot(y)
z1.backward()
print('After  first  backward:', x.grad)

z2 =  x.dot(y)
z2.backward()
print('After  second  backward:', x.grad)




# 中间节点默认不会保存梯度信息
q = x.dot(y)
z = q.sin()
z.backward()

with  warnings.catch_warnings(record=True) as warns:
    print('q.grad:', q.grad)
    for warn in warns:
        print("UserWarning:",warn.message)
#如果要查看中间节点的梯度,可以使用retain_grad()

q =  x.dot(y)
q.retain_grad()

z = q.sin()
z.backward()

print('q.grad:', q.grad)


# 原地操作破坏反向传播

z = x.dot(y)

try:
    x.relu_()
except RuntimeError as err:
    print("RuntimeError", err)
z = x.dot(y)
x = x.relu_()
z.backward()

x.grad: None
After  first  backward: tensor([5., 6., 7., 8.])
After  second  backward: tensor([10., 12., 14., 16.])
q.grad: None
q.grad: tensor(0.6333)
RuntimeError a leaf Variable that requires grad is being used in an in-place operation.


RuntimeError: a leaf Variable that requires grad is being used in an in-place operation.